#### Load data

In [11]:
import pandas as pd
import numpy as np

# =========================
# 1. Load data
# =========================
content = pd.read_csv("Data/Content.csv")
solutions = pd.read_csv("Data/Solutions.csv")

# =========================
# 2. Parse dates
# =========================
content["CreatedDate"] = pd.to_datetime(content["CreatedDate"], utc=True, errors="coerce")
content["LastAllowedDate"] = pd.to_datetime(content["LastAllowedDate"], utc=True, errors="coerce")
solutions["CreatedDate"] = pd.to_datetime(solutions["CreatedDate"], utc=True, errors="coerce")

# =========================
# 3. Define simulation date
# =========================
snapshot_date = pd.Timestamp("2025-11-01", tz="UTC")

# =========================
# 4. Filter content:
# only assignments (ContentType = 0)
# and only those available by snapshot date
# =========================
content_snapshot = content[
    (content["CreatedDate"] <= snapshot_date) &
    (content["ContentType"] == 0)
].copy()

# =========================
# 5. Filter solutions by date
# =========================
solutions_snapshot = solutions[
    solutions["CreatedDate"] <= snapshot_date
].copy()

# =========================
# 6. Keep only solutions for valid assignments
# =========================
content_cols = ["Id", "CourseId", "CreatedDate", "LastAllowedDate", "Name"]
content_small = content_snapshot[content_cols].copy()

merged = solutions_snapshot.merge(
    content_small,
    left_on="ContentId",
    right_on="Id",
    how="inner",
    suffixes=("_solution", "_content")
)

# =========================
# 7. Deduplicate:
# keep latest submission per student per assignment
# =========================
merged = merged.sort_values(["UserId", "ContentId", "CreatedDate_solution"])
merged_latest = merged.groupby(["UserId", "ContentId"], as_index=False).tail(1).copy()

# =========================
# 8. Convert grades
# =========================
merged_latest["Grade"] = pd.to_numeric(merged_latest["Grade"], errors="coerce")

# =========================
# 9. Build student list
# =========================
student_ids = sorted(merged_latest["UserId"].dropna().unique())
student_snapshot = pd.DataFrame({"student_id": student_ids})

# =========================
# 10. Total assignments
# =========================
total_assignments_available = content_small["Id"].nunique()
student_snapshot["total_assignments_available"] = total_assignments_available

# =========================
# 11. Submitted assignments
# =========================
submitted_counts = (
    merged_latest.groupby("UserId")["ContentId"]
    .nunique()
    .reset_index(name="submitted_assignments")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 12. Late submissions
# =========================
merged_latest["is_late"] = (
    merged_latest["LastAllowedDate"].notna() &
    (merged_latest["CreatedDate_solution"] > merged_latest["LastAllowedDate"])
)

late_counts = (
    merged_latest.groupby("UserId")["is_late"]
    .sum()
    .reset_index(name="late_submissions")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 13. Average grade
# =========================
avg_grade = (
    merged_latest.groupby("UserId")["Grade"]
    .mean()
    .reset_index(name="average_grade")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 14. Recent average grade (last 3)
# =========================
recent_grades = merged_latest.sort_values(
    ["UserId", "CreatedDate_solution"], ascending=[True, False]
).copy()

recent_grades["rank"] = recent_grades.groupby("UserId").cumcount() + 1
recent_top3 = recent_grades[recent_grades["rank"] <= 3]

recent_avg_grade = (
    recent_top3.groupby("UserId")["Grade"]
    .mean()
    .reset_index(name="recent_average_grade")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 15. Last submission + recency
# =========================
last_submission = (
    merged_latest.groupby("UserId")["CreatedDate_solution"]
    .max()
    .reset_index(name="last_submission_date")
    .rename(columns={"UserId": "student_id"})
)

last_submission["days_since_last_submission"] = (
    snapshot_date - last_submission["last_submission_date"]
).dt.days

# =========================
# 16. Assemble snapshot
# =========================
student_snapshot = student_snapshot.merge(submitted_counts, on="student_id", how="left")
student_snapshot = student_snapshot.merge(late_counts, on="student_id", how="left")
student_snapshot = student_snapshot.merge(avg_grade, on="student_id", how="left")
student_snapshot = student_snapshot.merge(recent_avg_grade, on="student_id", how="left")
student_snapshot = student_snapshot.merge(last_submission, on="student_id", how="left")

# =========================
# 17. Fill missing values
# =========================
student_snapshot["submitted_assignments"] = student_snapshot["submitted_assignments"].fillna(0).astype(int)
student_snapshot["late_submissions"] = student_snapshot["late_submissions"].fillna(0).astype(int)

student_snapshot["average_grade"] = student_snapshot["average_grade"].round(2)
student_snapshot["recent_average_grade"] = student_snapshot["recent_average_grade"].round(2)

# =========================
# 18. Submission rate
# =========================
student_snapshot["submission_rate"] = (
    student_snapshot["submitted_assignments"] / student_snapshot["total_assignments_available"]
).round(3)

# =========================
# 19. Final order
# =========================
student_snapshot = student_snapshot[
    [
        "student_id",
        "total_assignments_available",
        "submitted_assignments",
        "submission_rate",
        "late_submissions",
        "average_grade",
        "recent_average_grade",
        "last_submission_date",
        "days_since_last_submission",
    ]
]

student_snapshot = student_snapshot.sort_values("student_id").reset_index(drop=True)

# =========================
# 20. Preview
# =========================
print("Student snapshot rows:", len(student_snapshot))
print(student_snapshot.head(20))

Student snapshot rows: 30
    student_id  total_assignments_available  submitted_assignments  \
0            9                           18                     16   
1           10                           18                     16   
2           11                           18                     18   
3           12                           18                     18   
4           13                           18                     17   
5           14                           18                     17   
6           15                           18                     17   
7           16                           18                     17   
8           17                           18                     17   
9           18                           18                     17   
10          19                           18                     18   
11          20                           18                     17   
12          21                           18                     

In [15]:
# =========================
# Load students (only learners)
# =========================
users = pd.read_csv("Data/Users.csv")

# Keep only students (UserTypeId = 1)
students = users[users["UserTypeId"] == 1].copy()

# Select needed columns
students = students[
    ["Id", "Email", "FirstName", "LastName"]
].copy()

# Rename for consistency
students = students.rename(columns={"Id": "student_id"})

# Optional: sort
students = students.sort_values("student_id").reset_index(drop=True)

# Preview
print("Students:", len(students))
print(students.head(10))

Students: 31
   student_id                        Email FirstName LastName
0           7      john_doe@pseudomail.com      John      Doe
1           9     dan_cohen@pseudomail.com       Dan    Cohen
2          10      noa_levy@pseudomail.com       Noa     Levy
3          11  itay_mizrahi@pseudomail.com      Itay  Mizrahi
4          12    yael_biton@pseudomail.com      Yael    Biton
5          13   omer_peretz@pseudomail.com      Omer   Peretz
6          14    maya_david@pseudomail.com      Maya    David
7          15  lior_shapira@pseudomail.com      Lior  Shapira
8          16  tomer_azulay@pseudomail.com     Tomer   Azulay
9          17     shir_katz@pseudomail.com      Shir     Katz


In [17]:
# =========================
# Merge student info (names)
# =========================
student_snapshot = student_snapshot.merge(
    students,
    on="student_id",
    how="left"
)

#### Pre-LLM

Low performance

In [18]:
# =========================
# Low performance detector
# Rule: average_grade < 50
# =========================
low_performance_students = (
    student_snapshot[student_snapshot["average_grade"] < 50]
    .copy()
    .sort_values(["average_grade", "recent_average_grade"], ascending=[True, True])
    .reset_index(drop=True)
)

print("Low performance students:", len(low_performance_students))
print(
    low_performance_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Low performance students: 5
  FirstName LastName  average_grade  recent_average_grade  submission_rate  \
0      Itay  Mizrahi           8.78                  4.00            1.000   
1       Noa     Levy          10.44                  0.00            0.889   
2       Dan    Cohen          12.56                 16.33            0.889   
3       Adi    Levin          46.89                 21.67            1.000   
4      Amit    Cohen          48.88                 47.00            0.944   

   late_submissions  days_since_last_submission  
0                10                           0  
1                 7                           0  
2                 5                           2  
3                 5                           1  
4                 4                           0  


Declining performance

In [19]:
# =========================
# Declining performance detector
# Rule:
# recent_average_grade is at least 15 points lower than average_grade
# =========================
declining_performance_students = (
    student_snapshot[
        (student_snapshot["recent_average_grade"].notna()) &
        ((student_snapshot["average_grade"] - student_snapshot["recent_average_grade"]) >= 15)
    ]
    .copy()
)

declining_performance_students["grade_drop"] = (
    declining_performance_students["average_grade"] -
    declining_performance_students["recent_average_grade"]
).round(2)

declining_performance_students = (
    declining_performance_students
    .sort_values(["grade_drop", "recent_average_grade"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Declining performance students:", len(declining_performance_students))
print(
    declining_performance_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "grade_drop",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Declining performance students: 6
  FirstName  LastName  average_grade  recent_average_grade  grade_drop  \
0       Guy   Abramov          64.83                 34.67       30.16   
1        Or  Turgeman          91.00                 65.67       25.33   
2       Adi     Levin          46.89                 21.67       25.22   
3      Idan    Ohayon          66.56                 42.00       24.56   
4      Omer    Peretz          60.59                 38.67       21.92   
5      Maya     David          65.12                 47.67       17.45   

   submission_rate  late_submissions  days_since_last_submission  
0            1.000                 2                           0  
1            0.944                 1                           0  
2            1.000                 5                           1  
3            1.000                 2                           0  
4            0.944                 1                           1  
5            0.944                 2         

Low submission rate

In [33]:
# =========================
# Low engagement detector
# Rule: submission_rate < 0.90
# =========================
low_submission_students = (
    student_snapshot[student_snapshot["submission_rate"] < 0.8]
    .copy()
    .sort_values(
        ["submission_rate", "submitted_assignments", "late_submissions"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Low submission students:", len(low_submission_students))
print(
    low_submission_students[
        [
            "FirstName",
            "LastName",
            "submitted_assignments",
            "total_assignments_available",
            "submission_rate",
            "late_submissions",
            "average_grade",
            "recent_average_grade",
            "days_since_last_submission",
        ]
    ]
)

Low submission students: 0
Empty DataFrame
Columns: [FirstName, LastName, submitted_assignments, total_assignments_available, submission_rate, late_submissions, average_grade, recent_average_grade, days_since_last_submission]
Index: []


Top students

In [21]:
# =========================
# Top students detector
# Rule:
# average_grade >= 85
# =========================
top_students = (
    student_snapshot[student_snapshot["average_grade"] >= 85]
    .copy()
    .sort_values(
        ["average_grade", "recent_average_grade", "submission_rate"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print("Top students:", len(top_students))
print(
    top_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Top students: 3
  FirstName  LastName  average_grade  recent_average_grade  submission_rate  \
0     Sivan   Elkayam          95.59                 98.33            0.944   
1        Or  Turgeman          91.00                 65.67            0.944   
2       Nir     Hazan          85.44                 95.33            1.000   

   late_submissions  days_since_last_submission  
0                 0                           4  
1                 1                           0  
2                 2                           3  


#### LLM

Low Performance

In [26]:
# =========================
# Low performance LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
low_performance_for_llm = low_performance_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = low_performance_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as low performers.
Your task is NOT to find new students.
Your task is ONLY to classify the type of problem for each student.

Possible labels:
1. struggling_academically
   Use this when the student submits most assignments, but grades are very low.
   This suggests effort is present, but the student may not understand the material well.

2. disengaged
   Use this when the student shows lower engagement patterns, such as lower submission rate,
   many late submissions, or weak participation together with poor grades.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Do not call a student lazy or stupid.
- Use only the provided data.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "struggling_academically",
    "short_reason": "High submission rate but extremely low grades."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
low_performance_llm_output = llm_call(
    messages,
    max_new_tokens=400
)

print("Raw LLM output:")
print(low_performance_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_low_performance = safe_parse_llm_json(low_performance_llm_output)

print("\nParsed LLM output:")
print(parsed_low_performance)

# -------------------------
# Build memory with IDs
# -------------------------
memory_students = []

if parsed_low_performance is not None:
    for item in parsed_low_performance:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_students.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01",
            "signal_type": "low_performance"
        })

print("\nMemory entries:", len(memory_students))
print(memory_students[:5])

# -------------------------
# Save memory
# -------------------------
with open("memory_weak_students.json", "w", encoding="utf-8") as f:
    json.dump(memory_students, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_weak_students.json")

Raw LLM output:
[
  {
    "first_name": "Itay",
    "last_name": "Mizrahi",
    "label": "struggling_academically",
    "short_reason": "Perfect submission rate but very low recent grades."
  },
  {
    "first_name": "Noa",
    "last_name": "Levy",
    "label": "disengaged",
    "short_reason": "Lower submission rate and zero recent grades."
  },
  {
    "first_name": "Dan",
    "last_name": "Cohen",
    "label": "unclear",
    "short_reason": "Good submission rate and improving grades, mixed pattern."
  },
  {
    "first_name": "Adi",
    "last_name": "Levin",
    "label": "struggling_academically",
    "short_reason": "Perfect submission rate but very low recent grades."
  },
  {
    "first_name": "Amit",
    "last_name": "Cohen",
    "label": "struggling_academically",
    "short_reason": "High submission rate with low but consistent grades."
  }
]

Parsed LLM output:
[{'first_name': 'Itay', 'last_name': 'Mizrahi', 'label': 'struggling_academically', 'short_reason': 'Perfect submiss

Declining performance

In [24]:
# =========================
# Declining performance LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
declining_for_llm = declining_performance_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "grade_drop",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = declining_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as students with declining performance.
Your task is NOT to find new students.
Your task is ONLY to classify the type of decline for each student.

Possible labels:
1. temporary_drop
   Use this when the student's recent results are worse than usual, but the overall pattern still looks mostly stable.
   This suggests a temporary setback or short-term fluctuation.

2. worsening_trend
   Use this when the recent decline looks serious enough to suggest that the student may be entering a broader downward trend.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "worsening_trend",
    "short_reason": "Recent average is much lower than overall average, suggesting a meaningful decline."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
declining_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(declining_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_declining = safe_parse_llm_json(declining_llm_output)

print("\nParsed LLM output:")
print(parsed_declining)

# -------------------------
# Build memory with IDs
# -------------------------
memory_declining = []

if parsed_declining is not None:
    for item in parsed_declining:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_declining.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "declining_performance",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nDeclining memory entries:", len(memory_declining))
print(memory_declining[:5])

# -------------------------
# Save memory
# -------------------------
with open("memory_declining.json", "w", encoding="utf-8") as f:
    json.dump(memory_declining, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_declining.json")

Raw LLM output:
[
  {
    "first_name": "Guy",
    "last_name": "Abramov",
    "label": "worsening_trend",
    "short_reason": "Recent average grade dropped significantly from overall average, indicating a serious decline."
  },
  {
    "first_name": "Or",
    "last_name": "Turgeman",
    "label": "worsening_trend",
    "short_reason": "Recent average grade is substantially lower than overall average, suggesting a notable downward trend."
  },
  {
    "first_name": "Adi",
    "last_name": "Levin",
    "label": "worsening_trend",
    "short_reason": "Recent average grade is much lower than overall average, indicating a clear decline."
  },
  {
    "first_name": "Idan",
    "last_name": "Ohayon",
    "label": "worsening_trend",
    "short_reason": "Recent average grade dropped significantly compared to overall average, showing a serious decline."
  },
  {
    "first_name": "Omer",
    "last_name": "Peretz",
    "label": "worsening_trend",
    "short_reason": "Recent average grade is nota

Low submission

In [34]:
# =========================
# Low submission LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
low_submission_for_llm = low_submission_students[
    [
        "FirstName",
        "LastName",
        "submitted_assignments",
        "total_assignments_available",
        "submission_rate",
        "late_submissions",
        "average_grade",
        "recent_average_grade",
        "days_since_last_submission",
    ]
].copy()

student_table_text = low_submission_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as students with low homework submission.
Your task is NOT to find new students.
Your task is ONLY to classify the level/type of low engagement for each student.

Possible labels:
1. mild_low_engagement
   Use this when the student misses some assignments, but the pattern is not severe.
   This suggests a moderate engagement issue.

2. serious_low_engagement
   Use this when the student shows a clearly concerning engagement pattern,
   such as notably reduced submission rate, many late submissions, or weak participation signals.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "serious_low_engagement",
    "short_reason": "Submission rate is low and the student also has multiple late submissions."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
low_submission_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(low_submission_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_low_submission = safe_parse_llm_json(low_submission_llm_output)

print("\nParsed LLM output:")
print(parsed_low_submission)

# -------------------------
# Build memory with IDs
# -------------------------
memory_low_submission = []

if parsed_low_submission is not None:
    for item in parsed_low_submission:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_low_submission.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "low_submission",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nLow submission memory entries:", len(memory_low_submission))
print(memory_low_submission[:5])

# -------------------------
# Save memory
# -------------------------
with open("memory_low_submission.json", "w", encoding="utf-8") as f:
    json.dump(memory_low_submission, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_low_submission.json")

Raw LLM output:
[]

Parsed LLM output:
[]

Low submission memory entries: 0
[]

Memory saved to memory_low_submission.json


Top students

In [35]:
# =========================
# Top students LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
top_students_for_llm = top_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = top_students_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as top students.
Your task is NOT to find new students.
Your task is ONLY to classify the type of strong performance for each student.

Possible labels:
1. consistently_excellent
   Use this when the student shows strong overall results and the pattern looks stable.

2. excellent_with_positive_trend
   Use this when the student is strong overall and the recent results suggest additional positive momentum or improvement.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "consistently_excellent",
    "short_reason": "High overall grades with stable strong recent performance."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
top_students_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(top_students_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_top_students = safe_parse_llm_json(top_students_llm_output)

print("\nParsed LLM output:")
print(parsed_top_students)

# -------------------------
# Build memory with IDs
# -------------------------
memory_top_students = []

if parsed_top_students is not None:
    for item in parsed_top_students:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_top_students.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "top_students",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nTop students memory entries:", len(memory_top_students))
print(memory_top_students[:5])

# -------------------------
# Save memory
# -------------------------
with open("memory_top_students.json", "w", encoding="utf-8") as f:
    json.dump(memory_top_students, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_top_students.json")

Raw LLM output:
[
  {
    "first_name": "Sivan",
    "last_name": "Elkayam",
    "label": "excellent_with_positive_trend",
    "short_reason": "High overall grades with recent improvement in average grade."
  },
  {
    "first_name": "Or",
    "last_name": "Turgeman",
    "label": "unclear",
    "short_reason": "Strong overall average but recent average grade dropped significantly."
  },
  {
    "first_name": "Nir",
    "last_name": "Hazan",
    "label": "excellent_with_positive_trend",
    "short_reason": "Good overall average with clear recent improvement in grades."
  }
]

Parsed LLM output:
[{'first_name': 'Sivan', 'last_name': 'Elkayam', 'label': 'excellent_with_positive_trend', 'short_reason': 'High overall grades with recent improvement in average grade.'}, {'first_name': 'Or', 'last_name': 'Turgeman', 'label': 'unclear', 'short_reason': 'Strong overall average but recent average grade dropped significantly.'}, {'first_name': 'Nir', 'last_name': 'Hazan', 'label': 'excellent_with